# Caricamento dataset e esplorazione informativa



In [1]:
import pandas as pd

X_train = pd.read_csv("train.csv")
X_test = pd.read_csv("test.csv")


print(X_train.shape)
print(X_train.isna().mean() * 100)
print(X_test.isna().mean() * 100)

(891, 12)
PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64
PassengerId     0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            20.574163
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.239234
Cabin          78.229665
Embarked        0.000000
dtype: float64


Effettuiamo l'imputazione delle features 'Age' e 'Embarked' per X_train e 'Age' e 'Fare' per X_test, mentre rimuoviamo la feature 'Cabin' poiché troppo sparsa

In [ ]:
y_train = X_train["Survived"]
X_train = X_train.drop(columns=["Cabin", "Survived"])
X_test = X_test.drop(columns="Cabin")

# Effettuiamo un'imputazione per Age tramite group_by

# Per X_train facciamo transform sul train set
# transform restituisce una Series con le stesse dimensioni e indice della X_train
train_age_medians = X_train.groupby(['Pclass','Sex'])['Age'].transform('median') # Divide X_train nelle combinazioni possibili
X_train['Age'] = X_train['Age'].fillna(train_age_medians)

# Poiché sul test_set non possiamo fare transform, sennò sarebbe Data Leakage, 
# le mediane devono essere estratte dal training set

# Estraiamo la tabella di lookup dal train_set
median_map = X_train.groupby(['Pclass', 'Sex'])['Age'].median()


# Applichiamo la mappa alle righe di X_test, leggiamo la sua (Pclass,Sex) per ogni riga 
# e prendiamo il valore corrispondente dalla mappa  
test_age_imputed = X_test.set_index(['Pclass','Sex']).index.map(median_map)

# Riempiamo i valori mancanti 
X_test['Age'] = X_test['Age'].fillna(pd.Series(test_age_imputed, index=X_test.index))

KeyError: 'Survived'

In [5]:
print(median_map)
print(test_age_imputed)

Pclass  Sex   
1       female    41.0
        male      42.0
2       female    24.0
        male      28.0
3       female    22.0
        male      24.0
Name: Age, dtype: float64
Index([24.0, 22.0, 28.0, 24.0, 22.0, 24.0, 22.0, 28.0, 22.0, 24.0,
       ...
       22.0, 22.0, 22.0, 41.0, 22.0, 24.0, 41.0, 24.0, 24.0, 24.0],
      dtype='float64', length=418)


In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

cat_features = X_train.select_dtypes(include=['bool', 'object', 'string']).columns
num_features = X_train.select_dtypes(include=['number']).columns

cat_pipeline = Pipeline(steps=[
    ("imp", SimpleImputer(strategy='median')),
    ("enc", OneHotEncoder(handle_unknown='ignore'))
])